In [0]:
import java.nio.file._
import java.nio.file.attribute.BasicFileAttributes

val src = Paths.get("/Workspace/Repos/luisa.kamp@oge.net/pypsa-de-test")
val dst = Paths.get("/local_disk0/pypsa-de-test")

Files.walkFileTree(src, new SimpleFileVisitor[Path]() {

  override def preVisitDirectory(
      dir: Path,
      attrs: BasicFileAttributes
  ): FileVisitResult = {

    val relative = src.relativize(dir)

    // .pixi und scripts_LuKa komplett überspringen
    if (
      relative.toString == ".pixi" ||
      relative.toString == "scripts_LuKa"
    ) {
      println(s"Übersprungen: $relative")
      FileVisitResult.SKIP_SUBTREE
    } else {
      Files.createDirectories(dst.resolve(relative))
      FileVisitResult.CONTINUE
    }
  }

  override def visitFile(
      file: Path,
      attrs: BasicFileAttributes
  ): FileVisitResult = {

    val relative = src.relativize(file)

    Files.copy(
      file,
      dst.resolve(relative),
      StandardCopyOption.REPLACE_EXISTING
    )

    FileVisitResult.CONTINUE
  }
})

println("Repo nach /local_disk0/pypsa-de-test kopiert.")
println(s"pixi.toml vorhanden: ${Files.exists(dst.resolve("pixi.toml"))}")

In [0]:
import scala.sys.process._

// Bash-Skript: Installation und Umgebungsprüfung
val script = """
set -euxo pipefail

## 1. Pixi installieren, falls noch nicht vorhanden
export PATH="$HOME/.pixi/bin:$PATH"

if ! command -v pixi >/dev/null 2>&1; then
  curl -fsSL https://pixi.sh/install.sh | sh
fi

pixi --version

## 2. Lokales Arbeitsverzeichnis anlegen
mkdir -p /local_disk0/pypsa-de-test

## 4. Projektumgebung installieren
cd /local_disk0/pypsa-de-test
pixi install

## 5. Python und Paketversionen in der Projektumgebung pruefen
pixi run python -c '
import sys
import numpy
import geopandas

print("Python:", sys.executable)
print("NumPy:", numpy.__version__)
print("GeoPandas:", geopandas.__version__)
'

## 6. Snakemake-Version pruefen
pixi run snakemake --version

## 7. Szenariokonfigurationen erzeugen
pixi run snakemake -c1 build_scenarios

## 8. Alle Jahre nacheinander optimieren
for YEAR in 2025 2030 2035 2040 2045 2050; do

  TARGET="results/20261023_/SR27_S3/networks/base_s_27__none_${YEAR}.nc"

  echo "=========================================="
  echo "Optimierung ${YEAR}"
  echo "Target: ${TARGET}"
  echo "=========================================="

  PYTHONPATH="" pixi run snakemake -c8 --jobs 1 \
    "$TARGET" \
    --resources mem_mb=60000 \
    --set-resources solve_sector_network_myopic:mem_mb=60000

  echo "=== ${YEAR} erfolgreich abgeschlossen ==="

done

"""

// Ausgabe und Fehlermeldungen im Job anzeigen
val logger = ProcessLogger(
  line => println(s"OUT: $line"),
  line => println(s"ERR: $line")
)

// Bash-Skript genau einmal ausführen
val exitCode = Seq("bash", "-lc", script).!(logger)

// Bei einem Shell-Fehler auch den Databricks-Job fehlschlagen lassen
require(
  exitCode == 0,
  s"Umgebungsprüfung fehlgeschlagen: Exit-Code $exitCode"
)

## Parameter für Optimierungslauf